# 02 — NLP Preprocessing

Cleans CVE descriptions, runs spaCy NER, engineers keyword features, encodes CVSS metadata.

**Run order:**
1. Cell 1 — Mount Drive + set paths
2. Cell 2 — Install spaCy
3. Cell 3 — Define all functions (cleaning + feature extraction)
4. Cell 4 — Run preprocessing (smart: handles first run AND update runs)
5. Cell 5 — Final verification

---
**First time running:** processes all rows from scratch (~47 min for 200k rows)

**After live updater:** only processes NEW rows, appends to existing file (~5-15 min)

## Cell 1 — Mount Drive + set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

BASE      = '/content/drive/MyDrive/CVE_Project'
RAW       = f'{BASE}/raw_data'
PROCESSED = f'{BASE}/processed'
os.makedirs(PROCESSED, exist_ok=True)

# Load raw CSV
df_raw = pd.read_csv(f'{RAW}/cves_raw.csv')
print(f'Raw CSV loaded:     {len(df_raw)} rows')
print(f'Columns:            {list(df_raw.columns)}')

# Check if processed file already exists
if os.path.exists(f'{PROCESSED}/cves_processed.csv'):
    df_existing = pd.read_csv(f'{PROCESSED}/cves_processed.csv')
    already_done = set(df_existing['cve_id'].tolist())
    print(f'\nProcessed file found: {len(df_existing)} rows already done')
    print(f'New rows to process:  {len(df_raw) - len(already_done)}')
else:
    df_existing  = None
    already_done = set()
    print(f'\nNo processed file found. Will process all {len(df_raw)} rows from scratch.')

Mounted at /content/drive
Raw CSV loaded:     200431 rows
Columns:            ['cve_id', 'description', 'cvss_score', 'cvss_label', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope']

Processed file found: 200431 rows already done
New rows to process:  0


## Cell 2 — Install spaCy

In [2]:
!pip install spacy tqdm -q
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load('en_core_web_sm')
print('spaCy loaded successfully.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy loaded successfully.


## Cell 3 — Define all functions

In [3]:
import re
import pandas as pd

# ── Text cleaning ──────────────────────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'<.*?>', '', text)                    # remove HTML tags
    text = re.sub(r'cve-\d{4}-\d+', 'CVE_TOKEN', text)  # normalise CVE IDs
    text = re.sub(r'v?\d+\.\d+[\./\d]*', 'VERSION_TOKEN', text)  # normalise versions
    text = re.sub(r'[^a-z0-9\s_]', ' ', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()             # collapse whitespace
    return text

# ── Keyword feature flags ──────────────────────────────────────────────
REMOTE_WORDS   = ['remote', 'remotely', 'network']
UNAUTH_WORDS   = ['unauthenticated', 'unauthorized', 'no authentication']
EXEC_WORDS     = ['execute', 'execution', 'arbitrary code', 'command injection', 'rce']
PRIVESC_WORDS  = ['privilege escalation', 'root', 'admin', 'elevated privileges']
DOS_WORDS      = ['denial of service', 'dos', 'crash', 'unavailable']
OVERFLOW_WORDS = ['buffer overflow', 'heap overflow', 'stack overflow', 'out-of-bounds']

def extract_features(text):
    text_lower = text.lower()
    doc        = nlp(text_lower)
    entities   = [e.text for e in doc.ents if e.label_ in ['PRODUCT', 'ORG', 'GPE']]
    return {
        'entity_count':    len(entities),
        'entities':        ', '.join(entities[:5]),
        'has_remote':      int(any(w in text_lower for w in REMOTE_WORDS)),
        'has_unauth':      int(any(w in text_lower for w in UNAUTH_WORDS)),
        'has_exec':        int(any(w in text_lower for w in EXEC_WORDS)),
        'has_priv_esc':    int(any(w in text_lower for w in PRIVESC_WORDS)),
        'has_dos':         int(any(w in text_lower for w in DOS_WORDS)),
        'has_overflow':    int(any(w in text_lower for w in OVERFLOW_WORDS)),
        'desc_word_count': len(text.split())
    }

# ── Encoding maps ──────────────────────────────────────────────────────
ATTACK_VECTOR_MAP = {'NETWORK': 3, 'ADJACENT': 2, 'LOCAL': 1, 'PHYSICAL': 0}
COMPLEXITY_MAP    = {'LOW': 1, 'HIGH': 0}
PRIVS_MAP         = {'NONE': 2, 'LOW': 1, 'HIGH': 0}
UI_MAP            = {'NONE': 1, 'REQUIRED': 0}
SCOPE_MAP         = {'CHANGED': 1, 'UNCHANGED': 0}

def encode_metadata(df):
    df['attack_vector_enc']       = df['attack_vector'].map(ATTACK_VECTOR_MAP).fillna(0)
    df['attack_complexity_enc']   = df['attack_complexity'].map(COMPLEXITY_MAP).fillna(0)
    df['privileges_required_enc'] = df['privileges_required'].map(PRIVS_MAP).fillna(0)
    df['user_interaction_enc']    = df['user_interaction'].map(UI_MAP).fillna(0)
    df['scope_enc']               = df['scope'].map(SCOPE_MAP).fillna(0)
    return df

print('All functions defined.')

All functions defined.


## Cell 4 — Run preprocessing

**Smart mode — automatically detects which rows need processing:**
- First run → processes all rows from scratch
- After updater → only processes new rows, appends to existing file

In [4]:
from tqdm import tqdm
tqdm.pandas()

# ── Identify which rows need processing ───────────────────────────────
df_to_process = df_raw[~df_raw['cve_id'].isin(already_done)].copy()
df_to_process = df_to_process.reset_index(drop=True)

print(f'Total raw rows:      {len(df_raw)}')
print(f'Already processed:   {len(already_done)}')
print(f'Rows to process now: {len(df_to_process)}')

if len(df_to_process) == 0:
    print('\nNothing new to process. Everything is up to date.')
else:
    # Step 1 — Clean text
    print('\nStep 1/3 — Cleaning text...')
    df_to_process['description_clean'] = df_to_process['description'].apply(clean_text)
    print(f'  Done. Sample: {df_to_process["description_clean"].iloc[0][:80]}...')

    # Step 2 — NER + keyword features (slow step)
    print(f'\nStep 2/3 — Extracting NLP features ({len(df_to_process)} rows)...')
    print('  This is the slow step — ~14 min per 100k rows on Colab')
    features_df = df_to_process['description'].progress_apply(
        lambda x: pd.Series(extract_features(x))
    )
    df_to_process = pd.concat([df_to_process, features_df], axis=1)
    print(f'  Done. Shape: {df_to_process.shape}')

    # Step 3 — Encode metadata
    print('\nStep 3/3 — Encoding CVSS metadata...')
    df_to_process = encode_metadata(df_to_process)

    # Fix null entities
    df_to_process['entities'] = df_to_process['entities'].fillna('')
    print('  Done.')

    # ── Combine with existing and save ────────────────────────────────
    if df_existing is not None:
        print(f'\nAppending {len(df_to_process)} new rows to existing {len(df_existing)} rows...')
        df_final = pd.concat([df_existing, df_to_process], ignore_index=True)
    else:
        df_final = df_to_process

    df_final.to_csv(f'{PROCESSED}/cves_processed.csv', index=False)
    print(f'\nSaved {len(df_final)} rows to {PROCESSED}/cves_processed.csv')
    print(f'Total columns: {len(df_final.columns)}')
    print(f'Columns: {list(df_final.columns)}')

Total raw rows:      200431
Already processed:   200431
Rows to process now: 0

Nothing new to process. Everything is up to date.


## Cell 5 — Final verification

In [5]:
import pandas as pd

df_check  = pd.read_csv(f'{PROCESSED}/cves_processed.csv')
raw_count = len(pd.read_csv(f'{RAW}/cves_raw.csv'))

print(f'Processed rows:   {len(df_check)}')
print(f'Raw rows:         {raw_count}')
print(f'Rows match:       {len(df_check) == raw_count}')
print(f'Columns:          {len(df_check.columns)} (expected 24)')
print()

# Null check
nulls = df_check.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print('Nulls:            None')
else:
    print(f'Nulls found:\n{nulls}')

# Feature stats
print()
print('Feature stats:')
print(df_check[['has_remote','has_unauth','has_exec',
                'has_priv_esc','has_dos','has_overflow']].mean().round(3))

print()
if len(df_check) == raw_count and len(df_check.columns) == 24:
    print('All checks passed. Ready to run 04_embeddings.ipynb')
else:
    print('MISMATCH — check above and re-run Cell 4.')

Processed rows:   200431
Raw rows:         200431
Rows match:       True
Columns:          24 (expected 24)

Nulls found:
entities    144360
dtype: int64

Feature stats:
has_remote      0.261
has_unauth      0.111
has_exec        0.339
has_priv_esc    0.133
has_dos         0.105
has_overflow    0.079
dtype: float64

All checks passed. Ready to run 04_embeddings.ipynb
